- https://www.guruguru.science/competitions/27/discussions/7acf1579-ea62-4874-8c1f-7106cd13cdf8/
- https://www.guruguru.science/competitions/27/discussions/67cfa470-1990-4417-b3ec-797e931a0e9b/
- https://www.guruguru.science/competitions/27/discussions/6917111c-7262-4161-845c-2cd19995602f/

In [ ]:
from dataclasses import dataclass
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    PreTrainedTokenizerBase, 
    EvalPrediction,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)

from sklearn.model_selection import  GroupKFold
from sklearn.metrics import accuracy_score, roc_auc_score

In [ ]:
@dataclass
class Config:
    ver = 0
    n_splits = 5
    output_dir: str = "output"
    model_name: str = 'modernbert-ja-70m'
    checkpoint: str = "sbintuitions/modernbert-ja-30m" #
    max_length: int = 1024
    optim_type: str = "adamw_torch"
    per_device_train_batch_size: int = 4
    gradient_accumulation_steps: int = 4
    per_device_eval_batch_size: int = 8
    n_epochs: int = 10
    lr: float = 2e-4
    warmup_steps: int = 20
    seed = 2025


In [ ]:
config = Config()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(Config.checkpoint, trust_remote_code=False, use_fast=False)

In [ ]:
class CustomTokenizer:
    def __init__(
        self, 
        tokenizer: PreTrainedTokenizerBase, 
        max_length: int
    ) -> None:
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __call__(self, batch: dict) -> dict:
        tokenized = self.tokenizer(batch["prompt"], max_length=self.max_length, truncation=True)
        return {**tokenized, 
                "labels": batch["labels"]}

encode = CustomTokenizer(tokenizer, max_length=config.max_length)

In [ ]:
encode = CustomTokenizer(tokenizer, max_length=config.max_length)

In [ ]:
class TestCustomTokenizer:
    def __init__(
        self, 
        tokenizer: PreTrainedTokenizerBase, 
        max_length: int
    ) -> None:
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __call__(self, batch: dict) -> dict:
        tokenized = self.tokenizer(batch["prompt"], max_length=self.max_length, truncation=True)
        return {**tokenized}

In [ ]:
test_encode = TestCustomTokenizer(tokenizer, max_length=config.max_length)

In [ ]:
def compute_metrics(eval_preds: EvalPrediction) -> dict:
    logits = eval_preds.predictions
    labels = eval_preds.label_ids
    probs = torch.from_numpy(logits).float().softmax(-1).numpy()[:, 1]
    auc = roc_auc_score(labels, probs)
    acc = accuracy_score(y_true=labels, y_pred=probs > 0.5)
    return {"auc": auc, "acc": acc}

In [ ]:
def read_data():
    # 基本的なデータ
    train_df = pd.read_csv('../../data/raw/input/train.csv')
    test_df = pd.read_csv('../../data/raw/input/test.csv')
    submission_df = pd.read_csv('../../data/raw/input/sample_submission.csv')    
    return train_df, test_df, submission_df

train_df, test_df, submission_df = read_data()

test_ds = Dataset.from_pandas(test_df[['prompt']])
test_ds = test_ds.map(test_encode, batched=True)


In [ ]:
folds = np.zeros(len(train_df))
oof_preds = np.zeros(len(train_df))
test_preds = np.zeros(len(test_df))
kfold = GroupKFold(n_splits = config.n_splits, shuffle = True, random_state = config.seed)
for fold, (train_index, valid_index) in enumerate(kfold.split(train_df, train_df['labels'], train_df['社員番号'])):
    print(f'model training fold{fold+1}')
    train = train_df.iloc[train_index]
    valid = train_df.iloc[valid_index]
    folds[valid_index] = fold + 1

    steps = 2 * config.n_splits * int(len(train)/4*(config.n_splits-1)/config.n_splits/config.per_device_train_batch_size/config.gradient_accumulation_steps)
    training_args = TrainingArguments(
        output_dir=f'./models/{config.model_name}_VER{config.ver}_fold{fold+1}',
        overwrite_output_dir=True,
        report_to="none",
        num_train_epochs=config.n_epochs,
        per_device_train_batch_size=config.per_device_train_batch_size,
        gradient_accumulation_steps=config.gradient_accumulation_steps,
        per_device_eval_batch_size=config.per_device_eval_batch_size,
        eval_strategy="steps",
        logging_steps=steps,
        do_eval=True,
        eval_steps=steps,
        save_total_limit=1,
        save_strategy="steps",
        save_steps=steps,
        optim=config.optim_type,
        bf16=True, 
        learning_rate=config.lr,
        warmup_steps=config.warmup_steps,
        weight_decay=0.01,
        lr_scheduler_type='cosine',

        metric_for_best_model="auc",
        greater_is_better=True,
        seed=config.seed,
        data_seed=config.seed,
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        config.checkpoint,
        num_labels=2,
        torch_dtype=torch.bfloat16, 
        device_map="auto",
    )

    train_ds = Dataset.from_pandas(train[['prompt', 'labels']])
    valid_ds = Dataset.from_pandas(valid[['prompt', 'labels']])

    train_ds = train_ds.map(encode, batched=True)
    valid_ds = valid_ds.map(encode, batched=True)

    trainer = Trainer(
        args=training_args, 
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_ds,
        eval_dataset=valid_ds,
        compute_metrics=compute_metrics,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    )
    trainer.train()

    logits = trainer.predict(valid_ds).predictions  
    probs = torch.from_numpy(logits).float().softmax(-1).numpy()[:, 1]
    oof_preds[valid_index] = probs

    logits = trainer.predict(test_ds).predictions  
    probs = torch.from_numpy(logits).float().softmax(-1).numpy()[:, 1]
    test_preds += probs / config.n_splits

oof_df = pd.DataFrame()
oof_df['社員番号'] = train_df['社員番号']
oof_df['preds'] = oof_preds
oof_df['fold'] = folds
oof_df.to_csv(f"./oof/{config.model_name}_VER{config.ver}.csv", index=False)

preds_df = pd.DataFrame()
preds_df['社員番号'] = test_df['社員番号']
preds_df['preds'] = test_preds
preds_df.to_csv(f"./submissions/{config.model_name}_VER{config.ver}.csv", index=False)
preds_df['target'] = test_preds
preds_df[['target']].to_csv(f"./submissions/{config.model_name}_VER{config.ver}_submission.csv", index=False)

In [ ]:
# 評価関数の設定
def compute_metrics(eval_preds: EvalPrediction) -> dict:
    logits = eval_preds.predictions
    labels = eval_preds.label_ids
    probs = torch.from_numpy(logits).float().softmax(-1).numpy()[:, 1]
    auc = roc_auc_score(labels, probs)
    acc = accuracy_score(y_true=labels, y_pred=probs > 0.5)
    return {"auc": auc, "acc": acc}

for ...

   training_args = TrainingArguments(
        output_dir=f'./models/{config.model_name}_VER{config.ver}_fold{fold+1}',
        overwrite_output_dir=True,
        report_to="none",
        num_train_epochs=config.n_epochs,
        per_device_train_batch_size=config.per_device_train_batch_size,
        gradient_accumulation_steps=config.gradient_accumulation_steps,
        per_device_eval_batch_size=config.per_device_eval_batch_size,
        eval_strategy="steps",
        logging_steps=steps,
        do_eval=True,
        eval_steps=steps,
        save_total_limit=1,
        save_strategy="steps",
        save_steps=steps,
        optim=config.optim_type,
        bf16=True, 
        learning_rate=config.lr,
        warmup_steps=config.warmup_steps,
        weight_decay=0.01,
        lr_scheduler_type='cosine',
        metric_for_best_model="auc",
        greater_is_better=True,
        seed=config.seed,
        data_seed=config.seed,
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        config.checkpoint,
        num_labels=2,
        torch_dtype=torch.bfloat16, 
        device_map="auto",
    )

    trainer = CustomTrainer(
        args=training_args, 
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_ds,
        eval_dataset=valid_ds,
        compute_metrics=compute_metrics,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    )
    trainer.train()

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(config.checkpoint)
class CustomTokenizer:
    def __init__(
        self, 
        tokenizer: PreTrainedTokenizerBase, 
        max_length: int
    ) -> None:
        self.tokenizer = tokenizer
        self.max_length = max_length   
    def __call__(self, batch: dict) -> dict:
        tokenized = self.tokenizer(batch["prompt"], max_length=self.max_length, truncation=True)
        return {**tokenized, 
                "labels": batch["labels"]}  
encode = CustomTokenizer(tokenizer, max_length=config.max_length)
class TestCustomTokenizer:
    def __init__(
        self, 
        tokenizer: PreTrainedTokenizerBase, 
        max_length: int
    ) -> None:
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __call__(self, batch: dict) -> dict:
        tokenized = self.tokenizer(batch["prompt"], max_length=self.max_length, truncation=True)
        return {**tokenized}
test_encode = TestCustomTokenizer(tokenizer, max_length=config.max_length)